# Proteins, trajectories and conformer ensembles with DBSTEP 2.0

This notebook walks through the features added in DBSTEP 2.0:

1. steric parameters for one residue inside a protein (PDB input),
2. choosing what counts as the environment (waters, ligands, the residue itself),
3. running every residue and collecting a table,
4. splitting %V_bur between the residues that fill the sphere,
5. frames of a trajectory,
6. Boltzmann-weighted averages over a conformer ensemble.

It uses the small structures shipped with the test suite, so it runs in well under a minute. Every step shows the Python call and the equivalent command line.

In [ ]:
import os

import dbstep.Dbstep as db
from dbstep import ensemble, writer

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # repository root when run from examples/
PDB = os.path.join(ROOT, "tests", "pdb_files", "1a8o.pdb")        # HIV-1 capsid C-terminal domain, 70 residues, waters, no H
TRAJ = os.path.join(ROOT, "tests", "pdb_files", "ala5_traj.pdb")  # 10 models of a penta-alanine with a water moving in
SDF = os.path.join(ROOT, "tests", "sdf_files", "ether_conformers.sdf")  # 3 diethyl ether conformers from AQME with <Energy>

## 1. One residue in a protein

`residue="A:186"` selects chain A residue 186; `atom` names atom1 inside it (default CA) and `atom2` the Sterimol axis partner (default CB). Only the atoms within reach of the buried-volume sphere are kept (`cutoff="auto"` is switched on for you), so the calculation takes a fraction of a second even for a large protein.

Command line: `dbstep 1a8o.pdb --residue A:186 --vbur --sterimol --nowater`

In [ ]:
mol = db.dbstep(PDB, residue="A:186", volume=True, sterimol=True, nowater=True, quiet=True)
print(mol.residue_label, "| atoms kept:", mol.n_atoms_kept, "of", mol.n_atoms_total, "| cutoff %.2f A" % mol.cutoff)
print("%%V_bur = %.2f   L = %.2f   Bmin = %.2f   Bmax = %.2f" % (mol.bur_vol, mol.L, mol.Bmin, mol.Bmax))

## 2. What counts as the environment?

- `nowater=True` drops water molecules, `nohet=True` drops ligands and ions (modified residues such as MSE stay).
- `exclude_self=True` makes the residue itself occupy no volume: the result is the size of the pocket around it.
- `self_only=True` keeps only the residue, which is the same as measuring it extracted to its own file.

Command line flags have the same names: `--nowater --nohet --exclude-self --self-only`.

In [ ]:
variants = {
    "everything": {},
    "no water": {"nowater": True},
    "environment only (exclude self)": {"nowater": True, "exclude_self": True},
    "residue only": {"self_only": True},
}
print("%-34s %8s %8s" % ("", "%V_bur", "L"))
for name, flags in variants.items():
    m = db.dbstep(PDB, residue="A:186", volume=True, sterimol=True, quiet=True, **flags)
    print("%-34s %8.2f %8.2f" % (name, m.bur_vol, m.L))

## 3. Every residue at once

`all_residues` runs each polymer residue in turn and returns one object per residue. Each object carries `results`, the list of row dictionaries that the `--csv` option writes, so a whole-protein table is a one-liner.

Command line: `dbstep 1a8o.pdb --residue all --vbur --nowater --csv 1a8o_vbur.csv`

In [ ]:
runs = db.all_residues(PDB, volume=True, nowater=True, grid=0.1, quiet=True)
print(len(runs), "residues")
ranked = sorted(runs, key=lambda r: r.bur_vol, reverse=True)
for r in ranked[:5]:
    print("%-12s %6.2f" % (r.residue_label, r.bur_vol))

writer.csv_export("1a8o_vbur.csv", [row for r in runs for row in r.results])
print("wrote 1a8o_vbur.csv with", sum(len(r.results) for r in runs), "rows")

## 4. Which residues fill the sphere?

`decompose=True` splits %V_bur between the residues whose atoms occupy the sphere. A grid point covered by atoms of several residues is shared equally, so the contributions add up to the total exactly.

Command line: `dbstep 1a8o.pdb --residue A:186 --vbur --nowater --decompose --csv out.csv` (contributions go to `out_contributions.csv`).

In [ ]:
mol = db.dbstep(PDB, residue="A:186", volume=True, nowater=True, decompose=True, quiet=True)
contributions = sorted(mol.contributions.items(), key=lambda item: item[1], reverse=True)
for label, percent in contributions:
    if percent > 0.05:
        print("%-12s %6.2f" % (label, percent))
print("sum = %.2f, %%V_bur = %.2f" % (sum(mol.contributions.values()), mol.bur_vol))

try:
    import matplotlib.pyplot as plt

    labels, values = zip(*[(l, v) for l, v in contributions if v > 0.05])
    plt.figure(figsize=(6, 3))
    plt.bar(labels, values)
    plt.ylabel("%V_bur contribution")
    plt.xticks(rotation=45, ha="right")
    plt.title("Who fills the sphere around A:186 CA?")
    plt.tight_layout()
except ImportError:
    print("(install matplotlib to see the bar chart)")

## 5. Frames of a trajectory

Multi-MODEL PDB files, multi-frame xyz and multi-record sdf are treated as trajectories. `all_frames` runs the selected frames (`frames="::2"` takes every second one) and the `frame` column in `results` identifies them. Here a water approaches residue 3 over ten frames, so %V_bur rises unless waters are excluded.

Command line: `dbstep ala5_traj.pdb --residue A:3 --vbur --frames ::2 --csv traj.csv`

In [ ]:
with_water = db.all_frames(TRAJ, residue="A:3", volume=True, quiet=True)
without = db.all_frames(TRAJ, residue="A:3", volume=True, nowater=True, quiet=True)
print("frame  %V_bur   %V_bur (no water)")
for a, b in zip(with_water, without):
    print("%5d  %6.2f   %6.2f" % (a.results[0]["frame"], a.bur_vol, b.bur_vol))

## 6. Boltzmann averages over a conformer ensemble

A multi-record SDF from a conformer search carries an energy per record (here AQME's `<Energy>` field, in kcal/mol). `ensemble.boltzmann_average` turns the energies into populations, stores them on each run and returns the population-weighted parameters.

Command line: `dbstep ether_conformers.sdf --atom1 3 --atom2 2 --sterimol --vbur --boltzmann`

In [ ]:
conformers = db.all_frames(SDF, atom1=3, atom2=2, sterimol=True, volume=True, quiet=True)
summary = ensemble.boltzmann_average(conformers, temperature=298.15, units="kcal")
print("%-10s %8s %10s %7s %7s %7s" % ("conformer", "E", "population", "%V_bur", "Bmin", "L"))
for c in conformers:
    print("%-10s %8.3f %10.3f %7.2f %7.2f %7.2f" % (c.structure_name, c.energy, c.population, c.bur_vol, c.Bmin, c.L))
row = summary[0]
print("%-10s %8s %10.3f %7.2f %7.2f %7.2f" % ("boltzmann", "", row["population"], row["percent_vbur"], row["bmin"], row["L"]))

## Where to go next

- `dbstep --help` lists every option by group.
- `README.md` documents the residue, trajectory and ensemble options with worked examples.
- `docs/plans/2.0-proteins-and-trajectories.md` records the design and the tests behind these features.